In [1]:
import pandas as pd
import numpy as np

In [2]:
gazetteer = pd.read_csv("PUMS raw data files/gazette_data.txt",sep="|")


display(gazetteer.head())
print(gazetteer.columns.tolist())

,USPS,GEOID,GEOIDFQ,ANSICODE,NAME,LSAD,FUNCSTAT,ALAND,AWATER,ALAND_SQMI,AWATER_SQMI,INTPTLAT,INTPTLONG
0,NE,3100205,1600000US3100205,2397906,Abie village,47,A,240736,0,0.093,0.0,41.334104,-96.949747
1,NE,3100240,1600000US3100240,2397908,Adams village,47,A,1539268,0,0.594,0.0,40.459011,-96.507363
2,NE,3100380,1600000US3100380,2806910,Agnew CDP,57,S,488027,0,0.188,0.0,41.014234,-96.814270
3,NE,3100415,1600000US3100415,2393893,Ainsworth city,25,A,2600318,0,1.004,0.0,42.548439,-99.857522
4,NE,3100555,1600000US3100555,2393908,Albion city,25,A,3065278,0,1.184,0.0,41.687988,-97.999694


['USPS', 'GEOID', 'GEOIDFQ', 'ANSICODE', 'NAME', 'LSAD', 'FUNCSTAT', 'ALAND', 'AWATER', 'ALAND_SQMI', 'AWATER_SQMI', 'INTPTLAT', 'INTPTLONG']


In [3]:
#county seats we want to keep (also fixed harrisburg bc it wasnt returning a match)
county_seats = [
    "Alliance city",
    "Sidney city",
    "Chadron city",
    "Chappell city",
    "Oshkosh city",
    "Kimball city",
    "Bridgeport city",
    "Gering city",
    "Rushville city",
    "Harrison village",
    "Harrisburg CDP"]

#keeping only the columns we need
gazetteer_cleaned = gazetteer[["NAME", "INTPTLAT", "INTPTLONG"]].copy()
gazetteer_cleaned = gazetteer_cleaned[gazetteer_cleaned["NAME"].isin(county_seats)].copy()
gazetteer_cleaned = gazetteer_cleaned.rename(columns={
    "NAME": "Starting_City",
    "INTPTLAT": "Latitude",
    "INTPTLONG": "Longitude"})

display(gazetteer_cleaned)

,Starting_City,Latitude,Longitude
8,Alliance city,42.102567,-102.876188
74,Bridgeport city,41.664707,-103.097142
105,Chadron city,42.825975,-103.002478
110,Chappell city,41.091300,-102.469938
212,Gering city,41.826901,-103.662472
239,Harrisburg CDP,41.550529,-103.726684
240,Harrison village,42.688085,-103.882471
291,Kimball city,41.232433,-103.651862
406,Oshkosh city,41.408005,-102.345209
466,Rushville city,42.713380,-102.466563


In [4]:
#checking which county seats were not found
found_cities = gazetteer_cleaned["Starting_City"].tolist()

missing_cities = [
    city for city in county_seats
    if city not in found_cities]

print("Number of locations found:", len(found_cities))
print("Missing locations:", missing_cities)

Number of locations found: 11
Missing locations: []


In [5]:
# matching each starting city to its county
county_names = {
    "Harrisburg CDP": "BANNER",
    "Alliance city": "BOX BUTTE",
    "Sidney city": "CHEYENNE",
    "Chadron city": "DAWES",
    "Chappell city": "DEUEL",
    "Oshkosh city": "GARDEN",
    "Kimball city": "KIMBALL",
    "Bridgeport city": "MORRILL",
    "Gering city": "SCOTTS BLUFF",
    "Rushville city": "SHERIDAN",
    "Harrison village": "SIOUX"}

# adding the county column + organizingggg
gazetteer_cleaned["County"] = gazetteer_cleaned["Starting_City"].map(county_names)
gazetteer_cleaned = gazetteer_cleaned[["County", "Starting_City", "Latitude", "Longitude"]]
display(gazetteer_cleaned)


gazetteer_cleaned.to_csv("Agg Data FINAL/Panhandle_County_Locations.csv",index=False)

,County,Starting_City,Latitude,Longitude
8,BOX BUTTE,Alliance city,42.102567,-102.876188
74,MORRILL,Bridgeport city,41.664707,-103.097142
105,DAWES,Chadron city,42.825975,-103.002478
110,DEUEL,Chappell city,41.091300,-102.469938
212,SCOTTS BLUFF,Gering city,41.826901,-103.662472
239,BANNER,Harrisburg CDP,41.550529,-103.726684
240,SIOUX,Harrison village,42.688085,-103.882471
291,KIMBALL,Kimball city,41.232433,-103.651862
406,GARDEN,Oshkosh city,41.408005,-102.345209
466,SHERIDAN,Rushville city,42.713380,-102.466563


In [6]:
#census tract data
tracts = pd.read_csv("PUMS raw data files/gazette_updated.txt", sep="|", dtype={"GEOID": str})

panhandle_fips = {
    "007": "BANNER",
    "013": "BOX BUTTE",
    "033": "CHEYENNE",
    "045": "DAWES",
    "049": "DEUEL",
    "069": "GARDEN",
    "105": "KIMBALL",
    "123": "MORRILL",
    "157": "SCOTTS BLUFF",
    "161": "SHERIDAN",
    "165": "SIOUX"}

#county code from the GEOID
tracts["County_FIPS"] = tracts["GEOID"].str[2:5]

# filtering to the 11 panhandle counties
panhandle_tracts = tracts[tracts["County_FIPS"].isin(panhandle_fips)].copy()

panhandle_tracts["County"] = panhandle_tracts["County_FIPS"].map(panhandle_fips)

panhandle_tracts = panhandle_tracts[["County", "GEOID", "INTPTLAT", "INTPTLONG"]].copy()

panhandle_tracts = panhandle_tracts.rename(columns={"GEOID": "Tract_GEOID", "INTPTLAT": "Latitude", "INTPTLONG": "Longitude"})

display(panhandle_tracts.head())

print("Number of census tracts:", len(panhandle_tracts))
print(panhandle_tracts["County"].value_counts().sort_index())


,County,Tract_GEOID,Latitude,Longitude
13,BANNER,31007954000,41.539749,-103.726263
17,BOX BUTTE,31013951100,42.242769,-103.118882
18,BOX BUTTE,31013951200,42.140126,-102.766178
19,BOX BUTTE,31013951300,42.048946,-102.804311
50,CHEYENNE,31033954800,41.214236,-103.011929


Number of census tracts: 27
County
BANNER           1
BOX BUTTE        3
CHEYENNE         3
DAWES            2
DEUEL            1
GARDEN           1
KIMBALL          1
MORRILL          1
SCOTTS BLUFF    11
SHERIDAN         2
SIOUX            1
Name: count, dtype: int64


In [7]:
panhandle_tracts.to_csv("Agg Data FINAL/Panhandle_Census_Tracts.csv", index=False)